In [1]:
import sapphire

### general sapphire config 
config = {'num_cpus':50, # if running on CPU, set this to # cores (idiosyncrasy of jax), unnecessary for GPUs (which jax auto-detects)
           'tree_path':'/mnt/ceph/users/vpandya/jax_sapphire/tng_trees_jax/',
           'tree_type':'tng',
          'model':'jax_thermal',
          'coolfunc':'sd93',
            'output_path':'',
          'runtype':'inference' # single, multi, sampling, inference 
         }

# ODE solver parameters 
config['solver_config'] = {'engine':'diffrax', 
                         'rtol':1e-8, 
                         'atol':1e-8,
                         'rtol_atol':1e-8, # if assuming these are the same for convenience 
                        }

# parameters for physical model
# if you only want a single run, must set runtype == 'single' 
config['params_fixed_astro'] = {'alpha_n': -3/2., # slope of CGM density power law 
                              'alpha_T':0.0, # slope of CGM temperature power law 
                              'f_recycle':0.4, # instantaneous stellar-->ISM recycling fraction 
                              'yZ':0.02, # metal yield of 1 SN per 100 Msun of stars formed (2 Msun / 100 Msun = 0.02 for 10 Msun of SN ejecta) 

                                'A_M':0.0, 'alpha0_M':-1.0,'A_E':-1.0,'alpha0_E':-1.0,
                                'A_SF':0.8,'alpha0_SF':-1.8,'A_Z':-1.7,'alpha0_Z':-0.3,
                              "alphaz_M":0.0,
                              "beta_M":0.0,
                              "alphaz_E":0.0,
                              "beta_E":0.0,
                              "alphaz_SF":0.0,
                              "beta_SF":-0.7,
                              "alphaz_Z":0.0,
                              "beta_Z":0.0
        
                             }

# if you want to sample parameter space without inference -- ONLY if runtype=='sampling' above
config['sampling_config'] = {'method':'lhs', # none, lhs 
                               'rng_seed': 0, # int 
                               'Nsamples':100, # if on CPU or multi-GPU, MUST be equal to or an integer multiple of devices
                               # this simultaneously specifies which params are free and their bounds for sampling
                               'params_bounds':{"A_M":[-2.0,1.0],"alpha0_M":[-2.0,0.0],
                                                "A_E":[-2.0,0.0],"alpha0_E":[-2.0,0.0],
                                                "A_SF":[0.0,1.1],"alpha0_SF":[-2.0,0.0],
                                                "A_Z":[-2.0,0.0],"alpha0_Z":[-2.0,0.0]}
                               }


# add item for pairs of variables for univariate regression e.g., [M
# documentation should clarify -- different "types" may have different keys that need to be specified
### Maybe get rid of this for now 
config['summary_stats'] = {'type':'gaussian_kernel_regression', # none, gaussian_kernel_regression, kde 
                         # 'pairs':[(),(),()] # list of (x,y) = (independent,dependent) variable names
                          }


# if you want to run inference -- ONLY if runtype == 'inference' above 
config['inference_config'] = {'engine':'adam', # none, adam, hmc 
                              'backend':'manual', # numpyro, manual (likelihood for adam), blackjax 
                              'flag_smhm':1, # for lack of a better 
                              'flag_fgas':1, 
                              'flag_mzr':1,
                              'Nbatch':1000, # number halos per iteration; if on CPU or multi-GPU, must be integer multiple of devices
                              'Nchain':2, # number of chains for HMC/NUTS, or initial guesses for adam
                              'mock':True, # if True, will use mock_err below and sampling_config above
                              ### nested dict giving mock errors for each observable (this can be made more complicated internally)
                              'mock_err':{
                                  'smhm':{'constant':0.01, # "constant" in dex added in quadrature to intrinsic stderr(y)
                                          'scale':1.0 # multiplies intrinsic stderr(y)
                                         },
                                  'fgas':{'constant':0.01, # dex for log10(mism/mstar)
                                          'scale':1.0
                                         },
                                  'mzr':{'constant':0.01, # dex for log10(mzstar/mstar/zsun)
                                          'scale':1.0
                                         }
                              }                              
                             } 

sapphire.run(config)


your requested config:
 {'num_cpus': 50, 'tree_path': '/mnt/ceph/users/vpandya/jax_sapphire/tng_trees_jax/', 'tree_type': 'tng', 'model': 'jax_thermal', 'coolfunc': 'sd93', 'output_path': '', 'runtype': 'inference', 'solver_config': {'engine': 'diffrax', 'rtol': 1e-08, 'atol': 1e-08, 'rtol_atol': 1e-08}, 'params_fixed_astro': {'alpha_n': -1.5, 'alpha_T': 0.0, 'f_recycle': 0.4, 'yZ': 0.02, 'A_M': 0.0, 'alpha0_M': -1.0, 'A_E': -1.0, 'alpha0_E': -1.0, 'A_SF': 0.8, 'alpha0_SF': -1.8, 'A_Z': -1.7, 'alpha0_Z': -0.3, 'alphaz_M': 0.0, 'beta_M': 0.0, 'alphaz_E': 0.0, 'beta_E': 0.0, 'alphaz_SF': 0.0, 'beta_SF': -0.7, 'alphaz_Z': 0.0, 'beta_Z': 0.0}, 'sampling_config': {'method': 'lhs', 'rng_seed': 0, 'Nsamples': 100, 'params_bounds': {'A_M': [-2.0, 1.0], 'alpha0_M': [-2.0, 0.0], 'A_E': [-2.0, 0.0], 'alpha0_E': [-2.0, 0.0], 'A_SF': [0.0, 1.1], 'alpha0_SF': [-2.0, 0.0], 'A_Z': [-2.0, 0.0], 'alpha0_Z': [-2.0, 0.0]}}, 'summary_stats': {'type': 'gaussian_kernel_regression'}, 'inference_config': {'eng

In [ ]:
[ 0.  -1.  -1.  -1.   0.8 -1.8 -1.7 -0.3]

[-0.06926864 -0.93968982 -1.00249586 -1.03529385  0.81193769 -1.75953045
 -1.13964553 -0.24655706]

 [-0.01327513 -1.02048025 -0.99641826 -0.96363177  0.81484064 -1.76750152
 -1.66538632 -0.45250998]

In [1]:
import sapphire

### general sapphire config 
config = {'num_cpus':0, # if running on CPU, set this to # cores (idiosyncrasy of jax), unnecessary for GPUs (which jax auto-detects)
           'tree_path':'/mnt/ceph/users/vpandya/jax_sapphire/tng_trees_jax/',
           'tree_type':'tng',
          'model':'jax_thermal',
          'coolfunc':'sd93',
            'output_path':'',
          'runtype':'inference' # single, multi, sampling, inference 
         }

# ODE solver parameters 
config['solver_config'] = {'engine':'diffrax', 
                         'rtol':1e-8, 
                         'atol':1e-8,
                         'rtol_atol':1e-8, # if assuming these are the same for convenience 
                        }

# parameters for physical model
# if you only want a single run, must set runtype == 'single' 
config['params_fixed_astro'] = {'alpha_n': -3/2., # slope of CGM density power law 
                              'alpha_T':0.0, # slope of CGM temperature power law 
                              'f_recycle':0.4, # instantaneous stellar-->ISM recycling fraction 
                              'yZ':0.02, # metal yield of 1 SN per 100 Msun of stars formed (2 Msun / 100 Msun = 0.02 for 10 Msun of SN ejecta) 

                                'A_M':0.0, 'alpha0_M':-1.0,'A_E':-1.0,'alpha0_E':-1.0,
                                'A_SF':0.8,'alpha0_SF':-1.8,'A_Z':-1.7,'alpha0_Z':-0.3,
                              "alphaz_M":0.0,
                              "beta_M":0.0,
                              "alphaz_E":0.0,
                              "beta_E":0.0,
                              "alphaz_SF":0.0,
                              "beta_SF":-0.7,
                              "alphaz_Z":0.0,
                              "beta_Z":0.0
        
                             }

# if you want to sample parameter space without inference -- ONLY if runtype=='sampling' above
config['sampling_config'] = {'method':'lhs', # none, lhs 
                               'rng_seed': 0, # int 
                               'Nsamples':100, # if on CPU or multi-GPU, MUST be equal to or an integer multiple of devices
                               # this simultaneously specifies which params are free and their bounds for sampling
                               'params_bounds':{"A_M":[-2.0,1.0],"alpha0_M":[-2.0,0.0],
                                                "A_E":[-2.0,0.0],"alpha0_E":[-2.0,0.0],
                                                "A_SF":[0.0,1.1],"alpha0_SF":[-2.0,0.0],
                                                "A_Z":[-2.0,0.0],"alpha0_Z":[-2.0,0.0]}
                               }


# add item for pairs of variables for univariate regression e.g., [M
# documentation should clarify -- different "types" may have different keys that need to be specified
config['summary_stats'] = {'type':'gaussian_kernel_regression', # none, gaussian_kernel_regression, kde 
                         # 'pairs':[(),(),()] # list of (x,y) = (independent,dependent) variable names
                          }


# if you want to run inference -- ONLY if runtype == 'inference' above 
config['inference_config'] = {'engine':'adam', # none, adam, hmc 
                              'backend':'manual', # numpyro, manual (likelihood for adam), blackjax 
                              'flag_smhm':1, # for lack of a better 
                              'flag_fgas':1, 
                              'flag_mzr':1,
                              'Nbatch':1000, # number halos per iteration; if on CPU or multi-GPU, must be integer multiple of devices
                              'Nchain':2, # number of chains for HMC/NUTS, or initial guesses for adam
                              'mock':True, # if True, will use sampling_config above, rng_seed and params_bounds above  
                              ### nested dict giving mock errors for each observable (this can be made more complicated internally)
                              'mock_err':{
                                  'smhm':{'constant':0.01, # "constant" in dex added in quadrature to intrinsic stderr(y)
                                          'scale':1.0 # multiplies intrinsic stderr(y)
                                         },
                                  'fgas':{'constant':0.01, 
                                          'scale':1.0
                                         },
                                  'mzr':{'constant':0.01, 
                                          'scale':1.0
                                         }
                              }
                             } 


sapphire.run(config)


your requested config:
 {'num_cpus': 0, 'tree_path': '/mnt/ceph/users/vpandya/jax_sapphire/tng_trees_jax/', 'tree_type': 'tng', 'model': 'jax_thermal', 'coolfunc': 'sd93', 'output_path': '', 'runtype': 'inference', 'solver_config': {'engine': 'diffrax', 'rtol': 1e-08, 'atol': 1e-08, 'rtol_atol': 1e-08}, 'params_fixed_astro': {'alpha_n': -1.5, 'alpha_T': 0.0, 'f_recycle': 0.4, 'yZ': 0.02, 'A_M': 0.0, 'alpha0_M': -1.0, 'A_E': -1.0, 'alpha0_E': -1.0, 'A_SF': 0.8, 'alpha0_SF': -1.8, 'A_Z': -1.7, 'alpha0_Z': -0.3, 'alphaz_M': 0.0, 'beta_M': 0.0, 'alphaz_E': 0.0, 'beta_E': 0.0, 'alphaz_SF': 0.0, 'beta_SF': -0.7, 'alphaz_Z': 0.0, 'beta_Z': 0.0}, 'sampling_config': {'method': 'lhs', 'rng_seed': 0, 'Nsamples': 100, 'params_bounds': {'A_M': [-2.0, 1.0], 'alpha0_M': [-2.0, 0.0], 'A_E': [-2.0, 0.0], 'alpha0_E': [-2.0, 0.0], 'A_SF': [0.0, 1.1], 'alpha0_SF': [-2.0, 0.0], 'A_Z': [-2.0, 0.0], 'alpha0_Z': [-2.0, 0.0]}}, 'summary_stats': {'type': 'gaussian_kernel_regression'}, 'inference_config': {'engi

2025-11-26 18:07:38.668998: W external/xla/xla/service/gpu/nvptx_compiler.cc:836] The NVIDIA driver's CUDA version is 12.4 which is older than the PTX compiler version (12.6.20). Because the driver is older than the PTX compiler version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.


coeff_matrix.shape (108918, 5, 4, 97)
coeff_matrix.shape after halo mass cuts (107619, 5, 4, 97)
assigning random draw probs took 0.09 sec
rand_coeff_matrix.shape (10000, 5, 4, 97)
-22.59
params_arr
 [ 0.  -1.   0.   0.  -1.  -1.   0.   0.   0.8 -1.8  0.  -0.7 -1.7 -0.3
  0.   0.   0. ]
Ndevices for jax batching 1
diffrax shapes (10000, 5, 98) (10000, 5, 4, 97) (10000,)
vmapping batch_solve over single GPU for halos only
solving ODEs for runtype=inference...
[ 8.998052    9.68944709  9.57418855 56.84750537  6.66326213  7.40407039
  6.85981463]
initial jit+sol took 9.652 sec
summarizing mock data...
setting up model for inference...
param_samples
 [ 1.         -1.          0.          0.          0.1        -1.
  0.          0.          6.30957344 -1.8         0.         -0.7
  0.01995262 -0.3         0.          0.          0.        ]
test_params
 [ 0.  -1.  -1.  -1.   0.8 -1.8 -1.7 -0.3]
test_params_dict
 {'A_M': Array(0., dtype=float64), 'alpha0_M': Array(-1., dtype=float64), 'A_E':

Array([-0., -0., -0., -0., -0., -0., -0., -0.], dtype=float64)

In [ ]:
numpyro params_rand
 [ 0.1        -1.          0.          0.          0.1        -1.
  0.          0.          3.16227766 -1.          0.         -0.7
  0.1        -1.          0.          0.          0.        ]

In [ ]:
[ 1.         -1.          0.          0.          0.1        -1.
  0.          0.          6.30957344 -1.8         0.         -0.7
  0.01995262 -0.3         0.          0.          0.        ]

[-0. -0. -0. -0. -0. -0. -0. -0.]

In [ ]:
 [ 0.1        -1.          0.          0.          0.1        -1.
  0.          0.          3.16227766 -1.          0.         -0.7
  0.1        -1.          0.          0.          0.        ]